# Session 19: Correlation and Linear Regression

Data Science with Python, Level 1, Stage 3: Statistics and ML Fundamentals

## Learning Objectives

By the end of this session you will be able to:

1. Read a correlation coefficient and a correlation matrix.
2. Spot patterns that r misses.
3. Explain why correlation is not causation.
4. Fit and use a linear regression with scikit-learn.
5. Read the coefficients of a multiple regression.
6. Judge a model fairly on test data.

## Where We Are

So far you have described one variable at a time, or compared groups. Today you look at **pairs** of numeric variables. Do warmer days bring more bike rentals? Do students who study more score higher? The correlation coefficient puts a number on such a link, and you will see where that number misleads, including the big rule: correlation is not causation.

In the second half the link becomes a **prediction**. A linear regression draws the best line through the data and uses it to predict a target, such as the price of an apartment from its area. This is your first machine learning model, and the create, fit, predict pattern you learn today is the same for every model in the course.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(29)
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
exams = pd.read_csv(DATA_URL + "exam_scores.csv")
bikes = pd.read_csv(DATA_URL + "bike_rentals.csv")
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error

flats = pd.read_csv(DATA_URL + "apartments.csv")

## 1. The Correlation Coefficient

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 3))
x = rng.normal(0, 1, 300)
for ax, r in zip(axes, [0.95, 0.6, 0.0, -0.8]):
    y = r * x + np.sqrt(1 - r ** 2) * rng.normal(0, 1, 300)
    ax.scatter(x, y, s=6, alpha=0.5)
    ax.set_title(f"r = {np.corrcoef(x, y)[0, 1]:.2f}")
    ax.set_xticks([]); ax.set_yticks([])
plt.show()

| r | Strength | Direction |
| --- | --- | --- |
| 0.7 to 1.0 | Strong | Both rise together |
| 0.3 to 0.7 | Moderate | Both tend to rise together |
| -0.3 to 0.3 | Weak or none | No clear straight-line pattern |
| -0.7 to -0.3 | Moderate | One tends to fall as the other rises |
| -1.0 to -0.7 | Strong | One falls as the other rises |

These limits are rough guides, not rules. What counts as strong depends on the field.

## 2. Calculating Correlations

In [ ]:
r = exams["study_hours_week"].corr(exams["math"])
print("r =", round(r, 3))

r, p = stats.pearsonr(exams["study_hours_week"], exams["math"])
print(f"r = {r:.3f}, p-value = {p:.2e}")

In [ ]:
cols = ["study_hours_week", "sleep_hours", "attendance_pct",
        "math", "physics", "english"]
corr = exams[cols].corr().round(2)

sns.heatmap(corr, annot=True, cmap="RdBu_r", vmin=-1, vmax=1)
plt.title("Correlation matrix of the exam data")
plt.show()

## 3. What r Misses

In [ ]:
x = np.linspace(-3, 3, 200)
y = x ** 2 + rng.normal(0, 0.5, 200)
print("r for a U shape:", round(np.corrcoef(x, y)[0, 1], 3))

plt.scatter(x, y, s=8)
plt.title("A strong relationship with r close to 0")
plt.show()

In [ ]:
x = rng.normal(50, 10, 30)
y = rng.normal(50, 10, 30)
print("Without outlier:", round(np.corrcoef(x, y)[0, 1], 3))

x2, y2 = np.append(x, 150), np.append(y, 160)
print("With one outlier:", round(np.corrcoef(x2, y2)[0, 1], 3))
print("Spearman with outlier:", round(stats.spearmanr(x2, y2)[0], 3))

## 4. Correlation Is Not Causation

In [ ]:
bikes["month"] = pd.to_datetime(bikes["date"]).dt.month
r = bikes["humidity"].corr(bikes["rentals"])
print("Humidity and rentals:", round(r, 3))

print(bikes.groupby("weather")[["humidity", "rentals"]].mean().round(0))

In [ ]:
for w, g in bikes.groupby("weather"):
    print(w, round(g["humidity"].corr(g["rentals"]), 3))

## 5. What Is a Model?

In [ ]:
sns.scatterplot(data=flats, x="area_m2", y="price_k", s=12, alpha=0.5)
plt.title("Bigger apartments cost more")
plt.xlabel("Area (m2)")
plt.ylabel("Price (thousands)")
plt.show()

## 6. Simple Linear Regression

Every scikit-learn model works the same way: create it, `fit` it on the features and the target, then `predict`. The features `X` must be a table, even with one column, so we use double brackets.

The fitted line is `price = intercept + slope x area`. scikit-learn chooses the slope and intercept that make the squared prediction errors as small as possible; Session 20 shows how.

In [ ]:
X = flats[["area_m2"]]
y = flats["price_k"]

model = LinearRegression()
model.fit(X, y)

print("Slope:", round(model.coef_[0], 3))
print("Intercept:", round(model.intercept_, 2))

In [ ]:
new = pd.DataFrame({"area_m2": [60, 100, 150]})
print(model.predict(new).round(1))

xs = pd.DataFrame({"area_m2": np.linspace(30, 220, 50)})
plt.scatter(flats["area_m2"], flats["price_k"], s=10, alpha=0.4)
plt.plot(xs["area_m2"], model.predict(xs), color="tab:red", linewidth=2)
plt.title("The fitted line")
plt.show()

In [ ]:
flats["predicted"] = model.predict(X)
flats["residual"] = flats["price_k"] - flats["predicted"]

print(flats[["area_m2", "price_k", "predicted", "residual"]].head(4).round(1))
print("R2 on all data:", round(model.score(X, y), 3))

## 7. Multiple Regression

In [ ]:
features = ["area_m2", "rooms", "building_age", "distance_center_km", "elevator", "parking"]
X = flats[features]

model = LinearRegression().fit(X, y)
coefs = pd.Series(model.coef_, index=features).round(2)
print(coefs)
print("R2:", round(model.score(X, y), 3))

| Feature | Coefficient | Meaning, all else equal |
| --- | --- | --- |
| `area_m2` | about +0.84 | Each extra m2 adds about 840 |
| `building_age` | about -0.63 | Each year of age lowers the price by about 630 |
| `distance_center_km` | about -7.7 | Each km from the centre lowers it by about 7,700 |
| `elevator` | about +8.1 | An elevator adds about 8,100 |
| `parking` | about +8.6 | A parking space adds about 8,600 |

Prices are in thousands, so a coefficient of 0.84 means 840 per square metre.

## 8. Training and Test Data

A model that is judged on the same data it learned from can look better than it really is: it may have memorised details instead of learning the pattern. So we **split** the data: the model learns from the **training set** and is judged on the **test set**, which it never saw. `train_test_split` shuffles the rows and keeps 20 percent aside; `random_state` makes the split repeatable.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)
print(len(X_train), "training rows,", len(X_test), "test rows")

model = LinearRegression().fit(X_train, y_train)
pred = model.predict(X_test)

print("R2 on test data: ", round(r2_score(y_test, pred), 3))
print("MAE on test data:", round(mean_absolute_error(y_test, pred), 2))

In [ ]:
plt.scatter(y_test, pred, s=12, alpha=0.6)
lims = [y_test.min(), y_test.max()]
plt.plot(lims, lims, color="tab:red", linestyle="--")
plt.title("Predicted against actual prices (test set)")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.show()

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Overclaiming | Writing that A causes B because they correlate | Write that A is associated with B, and look for confounders. |
| Fake link | Trusting r driven by one outlier | Plot first, and check Spearman. |
| Too optimistic | Scoring the model on its own training data | Always report scores on a test set. |
| Leakage | Using the target, or something computed from it, as a feature | Features must be known before the price is. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Strongest Link

Which column of the exam data is most strongly correlated with the English score? Use `corr()` on all numeric columns and sort.

In [ ]:
# Write your code below

### In-Class Exercise 2: Confounder Hunt

Apartments further from the centre are cheaper. Is distance itself the reason, or is it the district? Compare the overall correlation of distance and price with the correlation inside each district.

In [ ]:
# Write your code below

### In-Class Exercise 3: Predict Your Flat

Using the multiple regression trained on the training set, predict the price of a 90 m2 flat with 3 rooms, 10 years old, 4 km from the centre, with an elevator and no parking.

In [ ]:
# Write your code below

### In-Class Exercise 4: Biggest Errors

Find the five test flats with the largest absolute prediction errors. What do they have in common?

In [ ]:
# Write your code below

## Summary

- r runs from -1 to 1 and measures the strength of a straight-line link.
- Always plot: curves and outliers fool r.
- Correlation is not causation; look for confounders.
- A model learns from features X to predict a target y.
- Every scikit-learn model follows create, fit, predict.
- Judge a model on test data it has never seen.

## What's Next

How does `fit` actually find the best line? Next session you will open the box and watch gradient descent learn step by step, then use the same idea to predict a category: will a customer leave or stay?